# 🤖 Multimodal RAG Chatbot — Local with Ollama + Gradio

### Text + Images + Tables from PDF / PPTX / DOCX

**Your System:** Python 3.11.9 | Ollama with llama3.2 + llava + mxbai-embed-large

---

### How It Works:
```
Step 1: Install packages
Step 2: Load AI models (local Ollama + sentence-transformers)
Step 3: Point to your files (no Google Colab upload needed)
Step 4: Extract text, images, tables from documents
Step 5: Chunk text semantically (page-aware)
Step 6: Build FAISS search indices
Step 7: Ask questions via Gradio web UI (ChatGPT-style)
```

### Models Used:
| # | Model | Role |
|---|---|---|
| 1 | `all-MiniLM-L6-v2` (sentence-transformers) | Text & table embedding (384-dim) |
| 2 | `clip-ViT-B-32` (sentence-transformers) | Image embedding (512-dim) |
| 3 | `cross-encoder/ms-marco-MiniLM-L-6-v2` | Reranker for accuracy |
| 4 | `llama3.2` (Ollama - local) | Answer generation |
| 5 | `llava` (Ollama - local, optional) | Image understanding |

---
## 🔧 Step 1: Install Dependencies
Run this cell ONCE. After that you can skip it.

In [1]:
# Install all required packages (run once)
!pip install -q pymupdf pdfplumber python-pptx python-docx faiss-cpu pillow \
    sentence-transformers gradio tabulate pandas ollama


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 🧠 Step 2: Import Libraries & Load AI Models

This loads 3 local embedding/reranking models via sentence-transformers.
Answer generation will use Ollama's llama3.2 (already on your system).

In [2]:
import os, io, re, base64, warnings, traceback
import numpy as np
import pandas as pd
from PIL import Image
import faiss
import fitz                              # PyMuPDF — PDF parsing
import pdfplumber                        # Table extraction from PDFs
from pptx import Presentation            # PowerPoint parsing
from pptx.enum.shapes import MSO_SHAPE_TYPE
from docx import Document as DocxDocument # Word DOC/DOCX parsing
from sentence_transformers import SentenceTransformer, CrossEncoder
import ollama
import gradio as gr

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

print(f"📦 Libraries imported. Gradio v{gr.__version__}")

📦 Libraries imported. Gradio v6.12.0


In [3]:
# ── MODEL 1: Text Embedding (384-dim) ──
# Converts text into vectors for semantic search
text_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model 1: all-MiniLM-L6-v2 (text+table embeddings, 384-dim)")

# ── MODEL 2: Image Embedding (512-dim) ──
# Maps images AND text into same vector space for cross-modal search
clip_model = SentenceTransformer('clip-ViT-B-32')
print("✅ Model 2: CLIP ViT-B/32 (image embeddings, 512-dim)")

# ── MODEL 3: Cross-Encoder Reranker ──
# Re-scores search results for higher accuracy
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Model 3: Cross-encoder reranker")

# ── MODEL 4: Ollama llama3.2 (already installed) ──
# Test that Ollama is running
try:
    test = ollama.chat(model='llama3.2', messages=[{'role':'user','content':'Hi'}])
    print("✅ Model 4: Ollama llama3.2 connected!")
except Exception as e:
    print(f"❌ Ollama not running! Start it with: ollama serve")
    print(f"   Error: {e}")

print("\n🎯 All models loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model 1: all-MiniLM-L6-v2 (text+table embeddings, 384-dim)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model 2: CLIP ViT-B/32 (image embeddings, 512-dim)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model 3: Cross-encoder reranker
✅ Model 4: Ollama llama3.2 connected!

🎯 All models loaded successfully!


---
## 📁 Step 3: Set Your File Paths

⚠️ **IMPORTANT:** Change the paths below to point to YOUR actual files.

You can add multiple files — PDF, PPTX, DOCX, PNG, JPG.

In [4]:
# ══════════════════════════════════════════════════════════════════════
# 👇 CHANGE THESE PATHS TO YOUR ACTUAL FILES
# ══════════════════════════════════════════════════════════════════════

uploaded_paths = [
    r"D:\LENOVO\Download\Sample.pdf",
    # r"D:\LENOVO\Download\another_file.pptx",  # uncomment to add more
    # r"D:\LENOVO\Download\report.docx",         # uncomment to add more
]

# ── Verify all files exist ──
valid_paths = []
for fpath in uploaded_paths:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f"   ✅ {os.path.basename(fpath)} ({size_mb:.1f} MB)")
        valid_paths.append(fpath)
    else:
        print(f"   ❌ NOT FOUND: {fpath}")
        print(f"      Check: Is the filename correct? Is it Download or Downloads?")

uploaded_paths = valid_paths
print(f"\n📁 {len(uploaded_paths)} file(s) ready to process.")

if not uploaded_paths:
    print("\n⚠️ No valid files! Fix the paths above and re-run this cell.")

   ✅ Sample.pdf (0.1 MB)

📁 1 file(s) ready to process.


---
## 📑 Step 4: Extract Text, Images & Tables (Page-Aware)

Every item is tagged with its page number so we can:
- Find which **pages** match a question
- Return images/tables only from those same pages

In [5]:
# ── Extraction Functions ──

def build_table_text_repr(df):
    """Convert DataFrame into searchable text description."""
    header = [str(h) for h in df.columns if str(h).strip()]
    parts = [f"Table with columns: {', '.join(header)}."]
    for row_idx, row in df.iterrows():
        vals = [f"{col}: {val}" for col, val in row.items()
                if str(val).strip() and str(val).strip() != 'None']
        if vals:
            parts.append(f"Row {row_idx+1} — " + "; ".join(vals))
    return " ".join(parts)


def extract_pdf(file_path):
    """Extract text, images, tables from PDF."""
    text_blocks, images, tables = [], [], []
    doc = fitz.open(file_path)
    for pn in range(len(doc)):
        page = doc[pn]
        text = page.get_text()
        if text.strip():
            text_blocks.append((pn, text.strip()))
        for img_info in page.get_images(full=True):
            try:
                base = doc.extract_image(img_info[0])
                pil_img = Image.open(io.BytesIO(base['image'])).convert('RGB')
                if pil_img.size[0] > 80 and pil_img.size[1] > 80:
                    images.append((pn, pil_img))
            except: continue
    doc.close()

    with pdfplumber.open(file_path) as pdf:
        for pn, page in enumerate(pdf.pages):
            for raw in page.extract_tables():
                if raw and len(raw) > 1:
                    cleaned = [[str(c).strip() if c else '' for c in r] for r in raw]
                    df = pd.DataFrame(cleaned[1:], columns=cleaned[0])
                    tables.append((pn, {'dataframe': df, 'text_repr': build_table_text_repr(df)}))
    return text_blocks, images, tables


def extract_pptx(file_path):
    """Extract text, images, tables from PowerPoint."""
    text_blocks, images, tables = [], [], []
    prs = Presentation(file_path)
    for sn, slide in enumerate(prs.slides):
        slide_texts = []
        for shape in slide.shapes:
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    t = para.text.strip()
                    if t: slide_texts.append(t)
            if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                try:
                    pil_img = Image.open(io.BytesIO(shape.image.blob)).convert('RGB')
                    if pil_img.size[0] > 80 and pil_img.size[1] > 80:
                        images.append((sn, pil_img))
                except: continue
            if shape.has_table:
                rows = [[cell.text.strip() for cell in row.cells] for row in shape.table.rows]
                if len(rows) > 1:
                    df = pd.DataFrame(rows[1:], columns=rows[0])
                    tables.append((sn, {'dataframe': df, 'text_repr': build_table_text_repr(df)}))
        if slide_texts:
            text_blocks.append((sn, '\n'.join(slide_texts)))
    return text_blocks, images, tables


def extract_docx(file_path):
    """Extract text, images, tables from Word document."""
    text_blocks, images, tables = [], [], []
    doc = DocxDocument(file_path)
    current_block, page_num = [], 0
    for i, para in enumerate(doc.paragraphs):
        t = para.text.strip()
        if t: current_block.append(t)
        if len(current_block) >= 20 or i == len(doc.paragraphs) - 1:
            if current_block:
                text_blocks.append((page_num, '\n'.join(current_block)))
                current_block = []
                page_num += 1
    for tbl in doc.tables:
        rows = [[cell.text.strip() for cell in row.cells] for row in tbl.rows]
        if len(rows) > 1:
            df = pd.DataFrame(rows[1:], columns=rows[0])
            tables.append((0, {'dataframe': df, 'text_repr': build_table_text_repr(df)}))
    for rel in doc.part.rels.values():
        if 'image' in rel.reltype:
            try:
                pil_img = Image.open(io.BytesIO(rel.target_part.blob)).convert('RGB')
                if pil_img.size[0] > 80 and pil_img.size[1] > 80:
                    images.append((0, pil_img))
            except: continue
    return text_blocks, images, tables


def extract_image_file(file_path):
    """Load standalone image file."""
    img = Image.open(file_path).convert('RGB')
    return [], [(0, img)], []


print("✅ Extraction functions defined (PDF, PPTX, DOCX, Image).")

✅ Extraction functions defined (PDF, PPTX, DOCX, Image).


In [6]:
# ── Run extraction on all files ──

all_text_blocks = []   # [(page, text), ...]
all_images = []        # [(page, PIL.Image), ...]
all_tables = []        # [(page, {dataframe, text_repr}), ...]

EXTRACTORS = {
    '.pdf': extract_pdf,
    '.pptx': extract_pptx, '.ppt': extract_pptx,
    '.docx': extract_docx, '.doc': extract_docx,
    '.png': extract_image_file, '.jpg': extract_image_file,
    '.jpeg': extract_image_file, '.bmp': extract_image_file,
}

for fpath in uploaded_paths:
    fname = os.path.basename(fpath)
    ext = os.path.splitext(fpath)[1].lower()
    print(f"\n📄 Processing: {fname}")
    extractor = EXTRACTORS.get(ext)
    if not extractor:
        print(f"   ⚠️ Unsupported format: {ext}")
        continue
    try:
        tb, imgs, tbls = extractor(fpath)
        all_text_blocks.extend(tb)
        all_images.extend(imgs)
        all_tables.extend(tbls)
        print(f"   ✅ Text: {len(tb)} | Images: {len(imgs)} | Tables: {len(tbls)}")
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Build page→content lookup
page_to_images = {}
for pg, img in all_images:
    page_to_images.setdefault(pg, []).append(img)

page_to_tables = {}
for pg, tbl in all_tables:
    page_to_tables.setdefault(pg, []).append(tbl)

print(f"\n{'='*60}")
print(f"📊 EXTRACTION COMPLETE")
print(f"   Text blocks : {len(all_text_blocks)}")
print(f"   Images      : {len(all_images)} (across {len(page_to_images)} pages)")
print(f"   Tables      : {len(all_tables)} (across {len(page_to_tables)} pages)")
print(f"{'='*60}")


📄 Processing: Sample.pdf
   ✅ Text: 4 | Images: 1 | Tables: 1

📊 EXTRACTION COMPLETE
   Text blocks : 4
   Images      : 1 (across 1 pages)
   Tables      : 1 (across 1 pages)


---
## ✂️ Step 5: Semantic Chunking (Page-Aware)

Splits text into meaningful chunks based on topic similarity, not just line breaks.

In [7]:
def split_into_sentences(text):
    raw = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [s.strip() for s in raw if len(s.strip()) > 25]

def semantic_chunk_with_pages(text_blocks_with_pages, sim_threshold=0.45, max_sent=8):
    chunks, chunk_pages = [], []
    for page_num, text in text_blocks_with_pages:
        sents = split_into_sentences(text)
        if not sents: continue
        if len(sents) == 1:
            if len(sents[0]) > 50:
                chunks.append(sents[0])
                chunk_pages.append(page_num)
            continue
        embs = text_model.encode(sents, normalize_embeddings=True, show_progress_bar=False)
        cur = [sents[0]]
        for i in range(1, len(sents)):
            sim = float(np.dot(embs[i], embs[i-1]))
            if sim >= sim_threshold and len(cur) < max_sent:
                cur.append(sents[i])
            else:
                ct = ' '.join(cur)
                if len(ct) > 50:
                    chunks.append(ct)
                    chunk_pages.append(page_num)
                cur = [sents[i]]
        if cur:
            ct = ' '.join(cur)
            if len(ct) > 50:
                chunks.append(ct)
                chunk_pages.append(page_num)
    return chunks, chunk_pages

print("✂️ Running semantic chunking...")
all_chunks, chunk_page_numbers = semantic_chunk_with_pages(all_text_blocks)
print(f"✅ {len(all_text_blocks)} text blocks → {len(all_chunks)} semantic chunks")

✂️ Running semantic chunking...
✅ 4 text blocks → 54 semantic chunks


---
## 🧮 Step 6: Build FAISS Search Indices

In [8]:
# ── Text Index (384-dim, cosine similarity) ──
text_index = None
if all_chunks:
    print("📝 Embedding text chunks...")
    text_embeddings = text_model.encode(all_chunks, normalize_embeddings=True, show_progress_bar=True)
    text_index = faiss.IndexFlatIP(text_embeddings.shape[1])
    text_index.add(np.array(text_embeddings).astype('float32'))
    print(f"   ✅ Text: {text_index.ntotal} vectors")

# ── Table Index (384-dim) ──
table_index = None
table_page_numbers = []
if all_tables:
    print("📊 Embedding tables...")
    table_texts = [t['text_repr'] for _, t in all_tables]
    table_page_numbers = [p for p, _ in all_tables]
    table_embs = text_model.encode(table_texts, normalize_embeddings=True, show_progress_bar=True)
    table_index = faiss.IndexFlatIP(table_embs.shape[1])
    table_index.add(np.array(table_embs).astype('float32'))
    print(f"   ✅ Tables: {table_index.ntotal} vectors")

# ── Image Index (512-dim, CLIP) ──
image_index = None
valid_images, image_page_numbers = [], []
if all_images:
    print("🖼️ Embedding images with CLIP...")
    img_embs = []
    for pg, img in all_images:
        try:
            emb = clip_model.encode(img)
            emb = emb / np.linalg.norm(emb)
            img_embs.append(emb)
            valid_images.append(img)
            image_page_numbers.append(pg)
        except: continue
    if img_embs:
        img_embs = np.array(img_embs).astype('float32')
        image_index = faiss.IndexFlatIP(img_embs.shape[1])
        image_index.add(img_embs)
        print(f"   ✅ Images: {image_index.ntotal} vectors")

print(f"\n🎯 INDICES READY — Text: {text_index.ntotal if text_index else 0} | "
      f"Tables: {table_index.ntotal if table_index else 0} | "
      f"Images: {image_index.ntotal if image_index else 0}")

📝 Embedding text chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ Text: 54 vectors
📊 Embedding tables...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ Tables: 1 vectors
🖼️ Embedding images with CLIP...
   ✅ Images: 1 vectors

🎯 INDICES READY — Text: 54 | Tables: 1 | Images: 1


---
## 💡 Step 7: Answer Generation with Ollama llama3.2

Uses your **local** llama3.2 model via Ollama.
- Multi-pass generation for detailed answers
- Strict "answer from context only" system prompt
- Out-of-document detection

In [9]:
def generate_answer(query, chunks, table_descs=None):
    """
    Generate detailed answer using Ollama llama3.2.
    Uses multi-pass for comprehensive responses.
    """
    # Build context from retrieved chunks
    context = '\n\n'.join(chunks[:7])
    if table_descs:
        context += '\n\nTable data: ' + ' '.join(table_descs[:2])

    # Truncate context to fit model window
    if len(context) > 6000:
        context = context[:6000]

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[
                {
                    'role': 'system',
                    'content': """You are a document assistant. Follow these rules STRICTLY:
1. Answer ONLY from the context provided below.
2. If the answer is NOT in the context, say exactly: "Not available in document."
3. Never use your own knowledge. Never guess.
4. Give DETAILED and COMPLETE answers — include all relevant steps, details, and explanations found in the context.
5. If the context contains step-by-step information, list ALL steps.
6. Use bullet points and clear formatting for readability."""
                },
                {
                    'role': 'user',
                    'content': f"""Context:
{context}

Question: {query}

Instructions: Give a detailed and complete answer using ALL relevant information from the context above.
If the answer is not in the context, say "Not available in document." only."""
                }
            ]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error generating answer: {str(e)}\n\nMake sure Ollama is running (ollama serve)."


print("✅ generate_answer() defined (uses Ollama llama3.2 locally).")

✅ generate_answer() defined (uses Ollama llama3.2 locally).


---
## 🔍 Step 8: Query Pipeline (Page-Aware Retrieval)

```
Question → FAISS text search (15 candidates)
         → Cross-encoder rerank (top 7)
         → Identify MATCHED PAGES
         → Tables from matched pages? → Include
         → Images from matched pages? → Include
         → Generate answer with Ollama
```

In [10]:
def query_rag(question):
    """
    Full RAG pipeline: retrieve → rerank → get page media → generate answer.
    Returns: (answer_text, result_tables, result_images)
    """
    q_emb = text_model.encode([question], normalize_embeddings=True).astype('float32')
    best_score = 0.0
    matched_pages = set()
    retrieved_chunks = []

    # ── TEXT RETRIEVAL + RERANKING ──
    if text_index and text_index.ntotal > 0:
        k = min(15, text_index.ntotal)
        D, I = text_index.search(q_emb, k=k)
        best_score = max(best_score, float(D[0][0]))

        candidates, cand_pages = [], []
        for j in range(len(I[0])):
            if float(D[0][j]) > 0.15:
                idx = int(I[0][j])
                candidates.append(all_chunks[idx])
                cand_pages.append(chunk_page_numbers[idx])

        # Cross-encoder reranking for accuracy
        if candidates:
            scores = reranker.predict([(question, c) for c in candidates])
            ranked = sorted(zip(scores, candidates, cand_pages),
                           key=lambda x: x[0], reverse=True)
            for sc, chunk, pg in ranked[:7]:
                if sc > -5:
                    retrieved_chunks.append(chunk)
                    matched_pages.add(pg)

    # ── OUT-OF-DOCUMENT CHECK ──
    if best_score < 0.20 and not matched_pages:
        return (
            "❌ This question is **not related** to the uploaded documents.\n\n"
            "I can only answer questions about the content in your files.\n\n"
            "**Suggestions:**\n"
            "• Try rephrasing using specific terms from the document\n"
            "• Ask about a topic you know is in the files\n"
            "• Upload additional documents that cover this subject"
        ), [], []

    # ── TABLES from matched pages only ──
    result_tables, table_descs = [], []
    for pg in matched_pages:
        if pg in page_to_tables:
            for tbl in page_to_tables[pg]:
                result_tables.append(tbl['dataframe'])
                table_descs.append(tbl['text_repr'][:500])
                if len(result_tables) >= 2: break
        if len(result_tables) >= 2: break

    # ── IMAGES from matched pages only ──
    result_images = []
    for pg in matched_pages:
        if pg in page_to_images:
            for img in page_to_images[pg]:
                result_images.append(img)
                if len(result_images) >= 2: break
        if len(result_images) >= 2: break

    # ── GENERATE ANSWER ──
    if retrieved_chunks:
        answer = generate_answer(question, retrieved_chunks,
                                  table_descs if table_descs else None)
    else:
        answer = "I found some related content but couldn't generate a detailed answer. Please try rephrasing."

    return answer, result_tables, result_images


print("✅ query_rag() defined (page-aware, tables/images only when relevant).")

✅ query_rag() defined (page-aware, tables/images only when relevant).


In [1]:
import gradio as gr

print(f"Gradio version: {gr.__version__}")
print(f"\n📊 Knowledge Base:")
print(f"   Text chunks: {text_index.ntotal if text_index else 0}")
print(f"   Tables: {table_index.ntotal if table_index else 0}")
print(f"   Images: {image_index.ntotal if image_index else 0}")


def pil_to_base64(img, max_size=500):
    """Convert PIL image to base64 for display in chat."""
    img_copy = img.copy()
    img_copy.thumbnail((max_size, max_size))
    buf = io.BytesIO()
    img_copy.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def on_send(message, history):
    """Handle user message → run RAG → return response."""
    question = (message or '').strip()
    if not question:
        return "Please type a question."

    answer, tables, images_out = query_rag(question)
    response = str(answer)

    if tables:
        response += '\n\n📊 **Relevant Table:**\n\n'
        for df in tables:
            response += f'```\n{df.to_string(index=False)}\n```\n'

    if images_out:
        response += '\n\n🖼️ **Relevant Image from Document:**\n\n'
        for img in images_out:
            b64 = pil_to_base64(img)
            response += (
                f'<img src="data:image/png;base64,{b64}" '
                f'style="max-width:480px; border-radius:8px; '
                f'border:1px solid #ddd;"/>\n'
            )

    return response


# ── Build Gradio UI ──
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="🤖 Document RAG Chatbot (Ollama + Local)"
) as demo:
    gr.Markdown(
        f"# 🤖 Document RAG Chatbot (Ollama + Local)\n\n"
        f"Ask questions about your uploaded documents. "
        f"Knowledge base: **{text_index.ntotal if text_index else 0}** text chunks, "
        f"**{table_index.ntotal if table_index else 0}** tables, "
        f"**{image_index.ntotal if image_index else 0}** images.\n\n"
        f"Powered by **Ollama llama3.2** (runs 100% locally on your machine)"
    )
    gr.ChatInterface(
        fn=on_send,
        examples=[
            "What is this document about?",
            "Summarize the key points in detail",
            "Explain the main concepts",
            "What are the important findings?",
            "List all the topics covered",
            "what is Nociceptors",
        ],
    )

# Launch!
demo.launch(share=False, debug=True)

Gradio version: 6.12.0

📊 Knowledge Base:


NameError: name 'text_index' is not defined

---
## 🌐 Step 9: Launch Gradio ChatGPT-Style Web UI

This creates a beautiful web interface at `http://localhost:7860`

Features:
- ChatGPT-style conversation
- Shows relevant images from documents
- Shows relevant tables from documents
- Out-of-document detection
- Example questions to get started

In [11]:
import gradio as gr

print(f"Gradio version: {gr.__version__}")
print(f"\n📊 Knowledge Base:")
print(f"   Text chunks: {text_index.ntotal if text_index else 0}")
print(f"   Tables: {table_index.ntotal if table_index else 0}")
print(f"   Images: {image_index.ntotal if image_index else 0}")


def pil_to_base64(img, max_size=500):
    """Convert PIL image to base64 for display in chat."""
    img_copy = img.copy()
    img_copy.thumbnail((max_size, max_size))
    buf = io.BytesIO()
    img_copy.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def on_send(message, history):
    """
    Handle user message → run RAG → return response.
    Works with Gradio ChatInterface.
    """
    question = (message or '').strip()
    if not question:
        return "Please type a question."

    # Run RAG pipeline
    answer, tables, images_out = query_rag(question)

    # Build response
    response = str(answer)

    if tables:
        response += '\n\n📊 **Relevant Table:**\n\n'
        for df in tables:
            response += f'```\n{df.to_string(index=False)}\n```\n'

    if images_out:
        response += '\n\n🖼️ **Relevant Image from Document:**\n\n'
        for img in images_out:
            b64 = pil_to_base64(img)
            response += (
                f'<img src="data:image/png;base64,{b64}" '
                f'style="max-width:480px; border-radius:8px; '
                f'border:1px solid #ddd;"/>\n'
            )

    return response


# ── Build Gradio UI ──
demo = gr.ChatInterface(
    fn=on_send,
    title="🤖 Document RAG Chatbot (Ollama + Local)",
    description=(
        f"Ask questions about your uploaded documents. "
        f"Knowledge base: **{text_index.ntotal if text_index else 0}** text chunks, "
        f"**{table_index.ntotal if table_index else 0}** tables, "
        f"**{image_index.ntotal if image_index else 0}** images.\n\n"
        f"Powered by **Ollama llama3.2** (runs 100% locally on your machine)"
    ),
    examples=[
        "What is this document about?",
        "Summarize the key points in detail",
        "Explain the main concepts",
        "What are the important findings?",
        "List all the topics covered",
    ],
    theme=gr.themes.Soft(),
    retry_btn="🔄 Retry",
    undo_btn="↩️ Undo",
    clear_btn="🗑️ Clear",
)

# Launch! Opens in your browser at http://localhost:7860
demo.launch(share=False, debug=True)

Gradio version: 6.12.0

📊 Knowledge Base:
   Text chunks: 54
   Tables: 1
   Images: 1


TypeError: ChatInterface.__init__() got an unexpected keyword argument 'theme'